見つからない

In [ ]:
import random
import numpy as np
import sys
import time
from scipy.sparse import csr_matrix, hstack, vstack

# --- 表示設定 ---
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

class BacktrackingGirthOptimizer:
    def __init__(self, P=768, J=3, L_half=6):
        self.P = P
        self.J = J
        self.L_half = L_half
        self.mid = P // 2
        self.rho_A = self._generate_random_cycle(range(0, self.mid))
        self.rho_B = self._generate_random_cycle(range(self.mid, self.P))
        self.zero_mat = csr_matrix((P, P), dtype=np.int8)

    def _generate_random_cycle(self, r):
        indices = list(r)
        random.shuffle(indices)
        p = list(range(self.P))
        for i in range(len(indices)):
            p[indices[i]] = indices[(i + 1) % len(indices)]
        return tuple(p)

    def get_power(self, base_p, k):
        res = list(range(self.P))
        curr = list(base_p)
        k %= self.mid
        while k > 0:
            if k % 2 == 1: res = [res[curr[i]] for i in range(self.P)]
            curr = [curr[curr[i]] for i in range(self.P)]
            k //= 2
        return tuple(res)

    def inject_swap(self, p, r):
        p_list = list(p)
        i1, i2 = random.sample(list(r), 2)
        p_list[i1], p_list[i2] = p_list[i2], p_list[i1]
        return tuple(p_list)

    def count_c4_partial(self, F_list, G_list):
        F_m = [tuple_to_sparse(f, self.P) for f in F_list] + [self.zero_mat] * (self.L_half - len(F_list))
        G_m = [tuple_to_sparse(g, self.P) for g in G_list] + [self.zero_mat] * (self.L_half - len(G_list))
        
        rows = []
        for i in range(self.J):
            row = [F_m[(j-i)%self.L_half] for j in range(self.L_half)] + \
                  [G_m[(j-i)%self.L_half] for j in range(self.L_half)]
            rows.append(hstack(row))
        Hx = vstack(rows)
        
        H_int = Hx.astype(np.int64)
        Gram = (H_int @ H_int.T).toarray()
        np.fill_diagonal(Gram, 0)
        return np.sum(Gram * (Gram - 1)) // 4

    def solve(self):
        start_time = time.time()
        print(f"=== 探索開始 (P={self.P}, J={self.J}, L={self.L_half*2}) ===")
        
        # Fブロックの探索
        F_final = self._backtrack_search([], [], "F")
        if not F_final:
            print("\nFブロックの解が見つかりませんでした。")
            return None
        
        print(f"\n[Success] Fブロック確定。Gブロックの探索へ移行します。 (経過時間: {time.time()-start_time:.2f}s)")
        
        # Gブロックの探索
        G_final = self._backtrack_search(F_final, [], "G")
        
        end_time = time.time()
        if G_final:
            print(f"\n=== 全探索完了！ 総所要時間: {end_time - start_time:.2f}秒 ===")
        return F_final, G_final

    def _backtrack_search(self, F_list, G_list, mode):
        current_depth = len(F_list) if mode == "F" else len(G_list)
        
        # 終了条件
        if current_depth == self.L_half:
            return F_list if mode == "F" else G_list

        indices = list(range(1, self.mid))
        random.shuffle(indices)
        
        total_candidates = len(indices)
        print(f"\n>>> {mode}[{current_depth}] の探索を開始 (候補数: {total_candidates})")

        for trial, k in enumerate(indices, 1):
            # 進捗を50件ごとに表示
            if trial % 50 == 0:
                sys.stdout.write(f"\r  {mode}[{current_depth}] 試行中: {trial}/{total_candidates} (k={k})")
                sys.stdout.flush()

            if mode == "F":
                test_p = self.get_power(self.rho_A, k)
                if current_depth == 0:
                    test_p = self.inject_swap(test_p, range(0, self.mid))
                
                # C4チェック
                if self.count_c4_partial(F_list + [test_p], G_list) == 0:
                    # 次の深さへ
                    res = self._backtrack_search(F_list + [test_p], G_list, "F")
                    if res is not None:
                        return res
            else:
                test_p = self.get_power(self.rho_B, k)
                if current_depth == 3: # G3に非可換ノイズ
                    noise = self.get_power(self.rho_A, random.randint(1, self.mid-1))
                    test_p = tuple(test_p[noise[m]] for m in range(self.P))
                if current_depth == 2:
                    test_p = self.inject_swap(test_p, range(self.mid, self.P))
                
                # C4チェック
                if self.count_c4_partial(F_list, G_list + [test_p]) == 0:
                    res = self._backtrack_search(F_list, G_list + [test_p], "G")
                    if res is not None:
                        return res
        
        # この深さで候補が尽きた場合
        print(f"\n  [Backtrack] {mode}[{current_depth}] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。")
        return None

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    return csr_matrix((np.ones(size, dtype=np.int8), (rows, p)), shape=(size, size))

# 実行
opt = BacktrackingGirthOptimizer(P=768, J=3)
result = opt.solve()

if result:
    F_res, G_res = result
    print("\n--- 全ブロックの確定に成功しました ---")
    print(f"F indices length: {len(F_res)}, G indices length: {len(G_res)}")

=== 探索開始 (P=768, J=3, L=12) ===

>>> F[0] の探索を開始 (候補数: 383)

>>> F[1] の探索を開始 (候補数: 383)

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=107)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=120)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=42))
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=316)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=268)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=300)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=105)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=313)
  [Backtrack] F[2] で適合する指数が見つかりませんでした。一つ前の段階に戻ります。

>>> F[2] の探索を開始 (候補数: 383)
  F[2] 試行中: 350/383 (k=312)
